# H1+ZEUS NC $F_2$ → pandas + minimal GCFT **xion** prior (illustrative)

Public data: **Aaron et al., JHEP 01 (2010) 109** — combined H1 and ZEUS inclusive NC $e^+p$ tables on [HEPData (ins836107)](https://www.hepdata.net/record/ins836107).

**Open in Colab:** upload this file or use *File → Upload notebook*, or open from GitHub via Colab’s GitHub tab once the notebook is in a public repo (`https://colab.research.google.com/` → GitHub → pick the `.ipynb`).

The **xion** block below is a **small multiplicative coherence correction** with Gaussian priors on its amplitude and position — a **scaffold** for a full GCFT fit, not a claim about the published GCFT manuscript.

In [ ]:
import io
import re
import tarfile
import gzip
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HEPDATA_CSV_TGZ = "https://www.hepdata.net/download/submission/ins836107/1/csv"

In [ ]:
def load_hepdata_ins836107_f2(url: str = HEPDATA_CSV_TGZ) -> pd.DataFrame:
    """Download HEPData CSV bundle (gzip-compressed tar) and return long-form F2 rows."""
    raw = urlopen(url).read()
    rows = []
    with tarfile.open(fileobj=gzip.GzipFile(fileobj=io.BytesIO(raw)), mode="r|") as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".csv"):
                continue
            text = tar.extractfile(member).read().decode("utf-8", errors="replace")
            q2 = None
            in_f2 = False
            for line in text.splitlines():
                line = line.strip()
                if not line:
                    continue
                m = re.match(r"#:\s*Q\*\*2\s*\[GeV\^2\]\s*,\s*([0-9.eE+-]+)", line)
                if m:
                    q2 = float(m.group(1))
                    in_f2 = False
                    continue
                if line.startswith("#"):
                    continue
                if line.upper().startswith("X,") and "F2" in line.upper() and "SIG" not in line.upper():
                    in_f2 = True
                    continue
                if in_f2 and q2 is not None:
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) < 4:
                        continue
                    try:
                        x = float(parts[0])
                        y = float(parts[1])
                        ecm = float(parts[2])
                    except ValueError:
                        continue
                    f2tok = parts[3].strip()
                    if f2tok in ("-", "", "nan"):
                        continue
                    try:
                        f2 = float(f2tok)
                    except ValueError:
                        continue
                    rows.append(
                        {"source": member.name, "x": x, "y": y, "Ecm": ecm, "Q2": q2, "F2": f2}
                    )
    return pd.DataFrame(rows)


df = load_hepdata_ins836107_f2()
df = df[np.isfinite(df["F2"]) & (df["F2"] > 0)].copy()
df["logx"] = np.log(df["x"])
df["logQ2"] = np.log(df["Q2"])
df["logF2"] = np.log(df["F2"])
print(len(df), "points")
df.head()

## GCFT **xion** prior injection (minimal)

Baseline: ordinary least squares in log space,
$\log F_2^{\rm base} = c_0 + c_1 \log x + c_2 \log Q^2$.

**Xion-shaped correction** (multiplicative, centred in $\log x$):
$$
F_2^{\rm pred} = F_2^{\rm base}\,\exp\bigl(\eta\,\phi(x)\bigr),\qquad
\phi(x) = \exp\!\left(-\frac{(\log(x/x_0))^2}{2w^2}\right).
$$

**Priors** (illustrative): $\eta \sim \mathcal{N}(0,\sigma_\eta^2)$, $\log_{10} x_0 \sim \mathcal{N}(\mu_{\log x0}, \tau^2)$, with fixed width $w$. Draw a few prior samples and compare residuals $(F_2^{\rm data}-F_2^{\rm pred})/F_2^{\rm data}$ vs $Q^2$.

In [ ]:
def fit_log_linear_baseline(d: pd.DataFrame):
    X = np.column_stack([np.ones(len(d)), d["logx"].values, d["logQ2"].values])
    y = d["logF2"].values
    coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return coef


coef = fit_log_linear_baseline(df)
c0, c1, c2 = coef


def log_f2_base(x, Q2):
    return c0 + c1 * np.log(x) + c2 * np.log(Q2)


def phi_xion(x, x0, w):
    return np.exp(-0.5 * (np.log(x / x0) / w) ** 2)


def f2_pred(x, Q2, eta, x0, w):
    lb = log_f2_base(x, Q2)
    return np.exp(lb + eta * phi_xion(x, x0, w))


rng = np.random.default_rng(42)
sigma_eta = 0.04
mu_log10_x0, tau_log10_x0 = -3.0, 0.35
w_xion = 1.15
n_draw = 400

etas = rng.normal(0.0, sigma_eta, size=n_draw)
log10_x0s = rng.normal(mu_log10_x0, tau_log10_x0, size=n_draw)
x0s = 10.0 ** log10_x0s

x = df["x"].values
Q2 = df["Q2"].values
f2 = df["F2"].values

res_samples = []
for eta, x0 in zip(etas, x0s):
    fp = f2_pred(x, Q2, eta, x0, w_xion)
    res_samples.append((f2 - fp) / f2)
res_samples = np.column_stack(res_samples)
res_med = np.median(res_samples, axis=1)
res_band = np.percentile(res_samples, [16, 84], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sc = ax.scatter(Q2, res_med, c=np.log10(x), s=12, alpha=0.75, cmap="viridis")
ax.errorbar(
    Q2,
    res_med,
    yerr=[res_med - res_band[0], res_band[1] - res_med],
    fmt="none",
    ecolor="0.5",
    alpha=0.25,
    elinewidth=0.6,
)
ax.axhline(0.0, color="k", lw=0.8, ls="--")
ax.set_xscale("log")
ax.set_xlabel(r"$Q^2\;(\mathrm{GeV}^2)$")
ax.set_ylabel(r"relative residual $(F_2^{\rm data}-F_2^{\rm pred})/F_2^{\rm data}$")
ax.set_title("H1+ZEUS combined NC $F_2$: baseline + xion prior samples (median + 68% band)")
cb = plt.colorbar(sc, ax=ax)
cb.set_label(r"$\log_{10}(x)$")
plt.tight_layout()
plt.show()